In [3]:
# download the helper files
# !wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py
# !wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
# !wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/04-evaluation/code/evaluation_utils.py

In [30]:
# import the required packages
from ingest import load_faq_data
from evaluation_utils import llm_structured, calc_price
from pydantic import BaseModel
from openai import OpenAI
from dotenv import load_dotenv
from pprint import pprint
import json
import os

In [ ]:
# load the environment variables
load_dotenv()
api_key = os.getenv('OPENAI_API_KEY')

In [ ]:
# fetch the faq data
documents = load_faq_data()
# filter for only 'llm-zoomcamp' documents
documents_llm = []
for doc in documents:
    if doc['course']=='llm-zoomcamp':
        documents_llm.append(doc)

documents = documents_llm 

print(f'Number of Documents: {len(documents)}')
pprint(f'Sample Document: {documents[0]}')
# 'id' becomes the label in our ground truth dataset - we generate questions from a document, so we know that this document holds the answer - later, search evaluation checks whether search brings 
# back the document with this id - this is why every record needs a stable id

Number of Documents: 103
("Sample Document: {'id': '74eb249bbf', 'course': 'llm-zoomcamp', 'section': "
 "'General Course-Related Questions', 'question': 'I just discovered the "
 "course. Can I still join?', 'answer': 'Yes, but if you want to receive a "
 'certificate, you need to submit your project while we’re still accepting '
 "submissions.'}")


In [24]:
# we ask the LLM to return data in a specific format using 'structured output' instead of free-form text - for example, instead of getting a paragraph that contains questions, we can ask for a 
# python object with a questions field
class Questions(BaseModel): # define a class 'Questions' that inherits from BaseModel class, which is pydantic's base class for data validation and parsing
    questions: list[str] # questions field that must be a list of strings
# pydantic will validate the input against type annotation, and raise a validation error or attempt to convert the values to strings, depending on the configuration
# this pattern is especially common when working with APIs (such as FastAPI) or structured outputs from llms, where you want to ensure the response contains exactly a list of strings under 
# the questions field.

In [13]:
# define the instructions for llm - ask the LLM to use different wording from the original document - thia will make the evaluation more realisitc
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [ ]:
# prepare a document as json
user_prompt = json.dumps(documents[0])
# create an llm client
llm_client = OpenAI(api_key=api_key)
# create the messages for llm request
messages = [
    {'role':'developer', 'content':data_gen_instructions},
    {'role':'user', 'content':user_prompt}
]

In [27]:
# call the model
response = llm_client.responses.parse(
    model='gpt-4o-mini',
    input=messages,
    text_format=Questions # text_format=Questions tells the API to return our class instead of free text
)

result = response.output_parsed
pprint(result.questions)

['I found out about this course late. Is it too late for me to enroll?',
 'Can I still join the course if I missed the start date?',
 'Is it possible to sign up now, even though the course has already started?',
 'If I enroll now, will I be able to get a certificate?',
 'What do I need to do to earn my certificate if I join at this point?']


In [32]:
# use the helper method with structured-output
result, usage = llm_structured(
    client=llm_client,
    instructions=data_gen_instructions,
    user_prompt=user_prompt,
    output_type=Questions,
    model='gpt-4o-mini'
)

pprint(f'Questions Generated: \n{result.questions}')
# track the token usage and cost
print(f'Cost: {calc_price(usage)}')

('Questions Generated: \n'
 "['I just found out about this course. Is it too late for me to enroll?', "
 "'Can I join the course now, or did I miss the opportunity?', 'If I sign up "
 "now, will I still be able to get a certificate?', 'Is there a deadline for "
 "submitting projects if I join the course late?', 'What happens if I enroll "
 "after the course has started? Will I still get a certificate?']")
Cost: {'input_cost': 0.00015675000000000002, 'output_cost': 0.0004005, 'total_cost': 0.00055725}


In [ ]:
# convert the questions to ground truth records
records = []
for q in result.questions:
    records.append({
        'question':q,
        'document':documents[0]['id']
    })

# each record has two fields - a) question: the question generated by the LLM, b) document: the id of the faq document that should answer the question
# the 'document' field connects the generated question to the document that contains the answer - later, when we evaluate search, we'll ask the search engine the generated question, and check if it 
# retrieves the document with this id
pprint(records)

[{'document': '74eb249bbf',
  'question': 'I just found out about this course. Is it too late for me to '
              'enroll?'},
 {'document': '74eb249bbf',
  'question': 'Can I join the course now, or did I miss the opportunity?'},
 {'document': '74eb249bbf',
  'question': 'If I sign up now, will I still be able to get a certificate?'},
 {'document': '74eb249bbf',
  'question': 'Is there a deadline for submitting projects if I join the '
              'course late?'},
 {'document': '74eb249bbf',
  'question': 'What happens if I enroll after the course has started? Will I '
              'still get a certificate?'}]
